In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

-- 1. Gold Monthly P&L & MoM Revenue Growth
CREATE OR REPLACE TABLE gold_monthly_pnl AS
WITH monthly_rollup AS (
  SELECT
    txn_month,
    ROUND(SUM(CASE WHEN account_type = 'Revenue' THEN amount_usd ELSE 0 END), 2) AS revenue_usd,
    ROUND(SUM(CASE WHEN account_type = 'COGS'    THEN amount_usd ELSE 0 END), 2) AS cogs_usd,
    ROUND(SUM(CASE WHEN account_type = 'OpEx'    THEN amount_usd ELSE 0 END), 2) AS opex_usd,
    COUNT(*) AS transaction_count
  FROM silver_transactions
  GROUP BY txn_month
)
SELECT
  txn_month,
  revenue_usd,
  cogs_usd,
  ROUND(revenue_usd - cogs_usd, 2) AS gross_profit_usd,
  ROUND((revenue_usd - cogs_usd) / NULLIF(revenue_usd, 0) * 100, 2) AS gross_margin_pct,
  opex_usd,
  ROUND(revenue_usd - cogs_usd - opex_usd, 2) AS net_income_usd,
  ROUND(
    (revenue_usd - LAG(revenue_usd) OVER (ORDER BY txn_month))
    / NULLIF(LAG(revenue_usd) OVER (ORDER BY txn_month), 0) * 100,
    2
  ) AS mom_revenue_growth_pct,
  transaction_count
FROM monthly_rollup
ORDER BY txn_month;

-- 2. Gold Departmental Budget vs. Actual Variance (Expenses Only)
CREATE OR REPLACE TABLE gold_budget_variance AS
WITH actual_expenses AS (
  SELECT
    txn_month,
    department,
    ROUND(SUM(amount_usd), 2) AS actual_spend_usd
  FROM silver_transactions
  WHERE account_type IN ('COGS', 'OpEx')
  GROUP BY txn_month, department
)
SELECT
  CAST(b.budget_month AS DATE) AS budget_month,
  b.department,
  b.budget_usd,
  COALESCE(a.actual_spend_usd, 0) AS actual_spend_usd,
  ROUND(COALESCE(a.actual_spend_usd, 0) - b.budget_usd, 2) AS variance_usd,
  ROUND((COALESCE(a.actual_spend_usd, 0) - b.budget_usd) / NULLIF(b.budget_usd, 0) * 100, 2) AS variance_pct,
  CASE
    WHEN COALESCE(a.actual_spend_usd, 0) > b.budget_usd THEN 'Over Budget'
    ELSE 'On Track'
  END AS budget_status
FROM bronze_budgets b
LEFT JOIN actual_expenses a
  ON CAST(b.budget_month AS DATE) = a.txn_month
  AND b.department = a.department
ORDER BY budget_month, department;

-- 3. Gold Audit Anomaly & Risk Register
CREATE OR REPLACE TABLE gold_audit_anomalies AS
SELECT
  txn_id,
  clean_date AS txn_date,
  department,
  account_id,
  account_name,
  vendor_name,
  amount_usd,
  z_score,
  CASE
    WHEN ABS(z_score) >= 4.5 THEN 'Critical'
    ELSE 'High'
  END AS risk_severity
FROM silver_transactions
WHERE is_anomaly = TRUE
ORDER BY ABS(z_score) DESC;